# The Instruments: Load Cells, Pressure Transducers, Torque Meters, and Multi-Axis Load Cells
### *Modern Strain Gage Technology* — Companion Notebook

**A calibration certificate is a promise about accuracy — and this notebook shows how that promise is measured: a straight line drawn through the data, then judged by the small errors that refuse to lie on it.**

Companion notebook to Chapter 44 — The Instruments: Load Cells, Pressure Transducers, Torque Meters, and Multi-Axis Load Cells.

A calibrated transducer is characterized by three performance specifications beyond sensitivity: *linearity*, *hysteresis*, and *repeatability*. These numbers — typically expressed as a percentage of full-scale output — set the systematic error budget of the measurement. Together they define the *combined error*, and the larger of hysteresis and linearity usually dominates.

*Linearity error* is the maximum deviation of the actual output from the best-fit straight line through the calibration data, expressed as a fraction of full scale. *Hysteresis* is the maximum difference in output for the same applied load when it is approached from above versus from below — a consequence of friction, internal damping, and in some cases plasticity in the spring element. This notebook generates a representative multi-point calibration dataset, fits the calibration line, and computes both errors directly from the data. The numbers here are deliberately large enough to be visible on a plot; a precision (sub-0.05% FS) load cell would show errors far too small to see at this scale.

**This notebook demonstrates:**
1. **Load cell calibration curve** — multi-point loading and unloading, best-fit line, linearity error
2. **Hysteresis analysis** — signed error at each force level and maximum hysteresis as % full scale

In [ ]:
# !pip install numpy matplotlib
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# Consistent, publication-style plot defaults
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.titlesize': 12})

# All notebook figures are written here; the path matches the \includegraphics
# calls in Chapter 44. Do not rename these output files.
OUT_DIR = Path("../src/media/ch44")
OUT_DIR.mkdir(parents=True, exist_ok=True)

---
## 1 — Calibration Curve and Linearity Error

A multi-point calibration applies known loads in ascending steps (*loading*) and then descending steps (*unloading*), recording the bridge output at each step. The best-fit line through the loading data defines the nominal sensitivity. Any deviation of the actual output from that line — expressed as a percentage of the full-scale output — is the *linearity error*.

For the data below the maximum linearity error comes out near 0.3% FS. That is intentionally larger than a real precision load cell — which would sit below 0.05% FS — so that the residual pattern is visible on the plot rather than lost in the thickness of the line. The loading and unloading curves are overlaid on the same axes to make the hysteresis visible: the gap between them at any given force level is the hysteresis error at that point.

In [ ]:
# Multi-point calibration data: applied reference loads and measured bridge outputs.
# Ascending steps form the loading branch; the descending steps are the unloading branch.
F_applied = np.array([0, 100, 200, 300, 400, 500, 400, 300, 200, 100, 0])      # N
V_output  = np.array([0.0, 2.01, 4.02, 6.05, 8.04, 10.00,
                      8.06, 6.08, 4.06, 2.03, 0.02])                            # mV

N_LOAD_STEPS = 6  # number of ascending points [0 ... 500 N]; the rest are unloading

F_load,   V_load   = F_applied[:N_LOAD_STEPS],     V_output[:N_LOAD_STEPS]
F_unload, V_unload = F_applied[N_LOAD_STEPS - 1:], V_output[N_LOAD_STEPS - 1:]


def fit_calibration(force, output):
    """Least-squares straight-line fit of bridge output vs. applied load.

    Returns (slope, intercept, fitted_output), where ``slope`` is the
    sensitivity in mV/N and ``fitted_output`` is the best-fit line evaluated
    at each value in ``force``.
    """
    slope, intercept = np.polyfit(force, output, 1)
    return slope, intercept, slope * force + intercept


def percent_full_scale(delta, full_scale):
    """Express a signed output difference as a percentage of full-scale output."""
    return delta / full_scale * 100.0


# Fit the loading branch; full-scale output is the fitted value at the top load.
sensitivity, intercept, V_fit = fit_calibration(F_load, V_load)
V_full_scale = V_fit[-1]

# Linearity error: residual of each loading point from the best-fit line, as % FS.
lin_err_pct = percent_full_scale(V_load - V_fit, V_full_scale)

# Hysteresis: unloading output vs. the loading curve at the same force levels.
V_unload_interp = np.interp(F_unload, F_load, V_load)
hyst_pct = percent_full_scale(np.max(np.abs(V_unload - V_unload_interp)), V_full_scale)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(F_load, V_load, 'o-', color='steelblue', lw=2, label='Loading')
axes[0].plot(F_unload, V_unload, 's--', color='tomato', lw=2, label='Unloading')
axes[0].plot(F_load, V_fit, 'k--', lw=1, alpha=0.5, label='Best fit')
axes[0].set_xlabel('Applied load (N)'); axes[0].set_ylabel('Output (mV)')
axes[0].set_title('Load Cell Calibration Curve'); axes[0].legend()
axes[0].grid(True, linestyle=':', alpha=0.5)

axes[1].plot(F_load, lin_err_pct, 'steelblue', lw=2, marker='o')
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_xlabel('Applied load (N)'); axes[1].set_ylabel('Linearity error (% FS)')
axes[1].set_title('Linearity Error'); axes[1].grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig44_nb1_load_cell_calibration.png', bbox_inches='tight', dpi=150)
plt.show()

print(f"Sensitivity: {sensitivity:.4f} mV/N  "
      f"({percent_full_scale(sensitivity, V_full_scale):.4f} %FS/N)")
print(f"Max linearity error: {np.max(np.abs(lin_err_pct)):.3f}% FS")
print(f"Max hysteresis: {hyst_pct:.3f}% FS")

---
## 2 — Hysteresis Analysis

Hysteresis is the difference between the output on loading and unloading at the same applied force. In a well-designed elastic spring element it arises primarily from internal material damping and microscopic slip at contact interfaces — it is *not* a sign that the element has yielded. Hysteresis that grows with cycle count, however, is a warning of accumulating damage.

The bar chart below shows the *signed* hysteresis error at each calibration point: a positive value means the unloading output is higher than the loading output (the element is recovering slowly from its deformed state). The maximum absolute value, expressed as a percent of full scale, is the hysteresis figure that appears on the calibration certificate. For the data here it reaches roughly 0.4% FS — again deliberately exaggerated so the effect is visible; a precision load cell would specify a value many times smaller.

In [ ]:
# Hysteresis at each shared force level: difference between the unloading and
# loading curves, in mV and as a percentage of full-scale output.
hyst_abs = V_unload - V_unload_interp                      # mV, signed
hyst_pct_pts = percent_full_scale(hyst_abs, V_full_scale)  # % FS at each point

BAR_WIDTH = 18  # N, visual width of each hysteresis bar

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: loading vs. unloading with the hysteresis band shaded between the curves.
axes[0].plot(F_load,  V_load,   'o-',  color='steelblue', lw=2, label='Loading')
axes[0].plot(F_unload, V_unload, 's--', color='tomato',    lw=2, label='Unloading')
axes[0].fill_betweenx(V_unload,
                      np.interp(V_unload, V_load, F_load),
                      F_unload,
                      alpha=0.15, color='tomato', label='Hysteresis band')
axes[0].set_xlabel('Applied load (N)')
axes[0].set_ylabel('Output (mV)')
axes[0].set_title('Hysteresis — Loading vs. Unloading')
axes[0].legend()
axes[0].grid(True, linestyle=':', alpha=0.5)

# Right: signed hysteresis error at each force level, as % FS.
axes[1].bar(F_unload, hyst_pct_pts, width=BAR_WIDTH, color='tomato',
            alpha=0.75, edgecolor='darkred')
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_xlabel('Applied load (N)')
axes[1].set_ylabel('Hysteresis error (% FS)')
axes[1].set_title(f'Hysteresis Error  [max = {np.max(np.abs(hyst_pct_pts)):.3f}% FS]')
axes[1].grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig44_nb1_hysteresis_measurement.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Where to take this next

The dataset above is a single, clean calibration cycle. A few concrete extensions turn it into a more realistic characterization tool:

1. **Add repeatability from repeated cycles.** Generate several loading/unloading cycles with a small amount of seeded random noise (`rng = np.random.default_rng(0)`), then compute repeatability as the spread of outputs at each force level. Combine linearity, hysteresis, and repeatability in quadrature to report a single *combined error* — the number that actually caps the certificate accuracy.
2. **Test the reference-line convention.** Compare a two-parameter fit (slope + intercept) against a forced-through-zero fit, and watch how the choice of reference line changes the reported linearity error. Calibration standards specify which convention applies, and the difference is not cosmetic.
3. **Model a real nonlinearity.** Replace the near-linear data with a diaphragm-style quadratic term (the large-deflection behavior discussed for pressure transducers in Chapter 44) and watch the residual plot bend into the characteristic bow-shaped linearity signature — then fit a polynomial correction and see how much accuracy it recovers.